# AI Vision Assistant - Indoor Object Detection Fine-Tuning
**Course**: B.E. CSE Mini-Project — Deep Learning
**Role**: Member A (Detection, Scene Understanding, Model Training)
**Objective**: Fine-tune YOLOv8 Nano (`yolov8n.pt`) on 10 Indoor Object classes for assistive navigation on CPU.

### Step 1: Check GPU Availability
Ensure you are connected to a GPU runtime in Colab (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install Ultralytics

In [ ]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

### Step 3: Mount Google Drive or Upload Dataset
Zip your local `training/indoor_objects` folder into `indoor_objects.zip` and upload it here (or to Google Drive).

In [ ]:
# Option A: Upload directly via Colab files pane or via python
# from google.colab import files
# uploaded = files.upload()

# Option B: Mount Google Drive (Recommended for speed)
from google.colab import drive
drive.mount('/content/drive')

# Unzip dataset to /content/indoor_objects
!mkdir -p /content/indoor_objects
!unzip -q "/content/drive/MyDrive/indoor_objects.zip" -d /content/indoor_objects/
!ls -la /content/indoor_objects

### Step 4: Verify and Update data.yaml
Make sure paths in `data.yaml` point correctly to the unzipped directories.

In [ ]:
import yaml

yaml_path = '/content/indoor_objects/data.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/content/indoor_objects/train/images'
data['val'] = '/content/indoor_objects/valid/images'
data['test'] = '/content/indoor_objects/test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Updated data.yaml:")
!cat /content/indoor_objects/data.yaml

### Step 5: Fine-tune YOLOv8n
We use pretrained `yolov8n.pt` weights and fine-tune for 25 epochs at image size 640.

In [ ]:
from ultralytics import YOLO

# Load pretrained model (COCO 80 classes)
model = YOLO('yolov8n.pt')

# Train on custom indoor dataset
results = model.train(
    data='/content/indoor_objects/data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    name='indoor_yolov8n',
    project='runs/detect',
    save=True,
    plots=True
)

### Step 6: Validate on Test Split

In [ ]:
val_metrics = model.val(data='/content/indoor_objects/data.yaml', split='test')
print(val_metrics)

### Step 7: Download Trained Weights & Results
Download `best.pt` (save locally as `models/yolov8n_indoor_finetuned.pt`)
and results plots (save locally into `training/training_results/` and `results/`).

In [ ]:
from google.colab import files

# Download the best fine-tuned model checkpoint
files.download('runs/detect/indoor_yolov8n/weights/best.pt')

# Zip and download all evaluation plots and metrics
!zip -r training_results.zip runs/detect/indoor_yolov8n/
files.download('training_results.zip')